# ByStander Fine-tuning (Colab)

Notebook นี้แยกโค้ดจาก `finetune.py` เป็นช่วง ๆ เพื่อรันบน Google Colab ได้ง่าย
- ใช้ Unsloth + LoRA
- ใช้ dataset: `bystander_augmented_gemini.jsonl`
- เทรนให้ output เป็น JSON: `guidance`, `severity`, `facility_type`

## 0) Runtime
บน Colab ให้เลือก **Runtime > Change runtime type > GPU** ก่อนเริ่ม

In [ ]:
# 1) Install dependencies (run once per session)
!pip -q install --upgrade pip
!pip -q install unsloth trl transformers datasets accelerate bitsandbytes sentencepiece


In [ ]:
# 2) Imports
import json
import os
from typing import Any, Dict, Optional, Tuple

from datasets import Dataset, load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel, is_bfloat16_supported


In [ ]:
# 3) Mount Google Drive (optional but recommended)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 4) Config
DATASET_PATH = '/content/drive/MyDrive/ByStander/bystander_augmented_gemini.jsonl'
BASE_MODEL = 'typhoon-ai/llama3.2-typhoon2-1b'
OUTPUT_DIR = '/content/drive/MyDrive/ByStander/outputs/bystander_sft'

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3.0
MAX_STEPS = -1  # >0 เพื่อ override epoch-based training
WARMUP_STEPS = 50
WEIGHT_DECAY = 0.01
EVAL_RATIO = 0.05
LOGGING_STEPS = 10
SAVE_STEPS = 100
EVAL_STEPS = 100
SEED = 3407
SAVE_GGUF = False

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('DATASET_PATH:', DATASET_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
# 5) Prompts and helpers
SYSTEM_PROMPT = (
    "You are the ByStander Emergency Intelligence Engine, a professional medical dispatcher "
    "specializing in Thai emergency protocols. Your goal is to provide immediate, "
    "stress-resistant, and factually perfect first-aid guidance. "
    "OPERATIONAL RULES: "
    "1. LANGUAGE: Use professional yet easy-to-understand Thai (Central dialect). "
    "2. TONE: Calm, authoritative, and instructional to minimize user panic. "
    "3. LOGIC: If the input is a MEDICAL/ACCIDENTAL emergency, categorize it as 'critical' or 'mild'. "
    "If the input is NOT an emergency, categorize it as 'none' and provide a helpful, brief advisory. "
    "4. SAFETY: Never provide instructions that require professional medical equipment unless specified in context. "
    "5. FORMAT: Output strictly in valid JSON with keys guidance, severity, facility_type. "
    "No Markdown formatting, no asterisks, no conversational filler."
)

def normalize_severity(v: Any) -> str:
    s = str(v or '').strip().lower()
    if s in {'critical', 'mild', 'none'}:
        return s
    return 'none'

def normalize_facility(v: Any) -> str:
    s = str(v or '').strip().lower()
    if s in {'hospital', 'clinic', 'none'}:
        return s
    return 'none'

def build_user_prompt(input_text: str) -> str:
    return (
        f'สถานการณ์: \"{input_text}\"\n'
        'ตอบเป็น JSON ที่มีฟิลด์: guidance, severity, facility_type. '
        'guidance: '
        "- หากเป็นเหตุฉุกเฉิน: เริ่มต้นด้วยประโยค 'สถานการณ์นี้เป็นเหตุฉุกเฉิน' "
        'ตามด้วยขั้นตอนการปฐมพยาบาลแบบ Step-by-Step ที่ละเอียดและถูกต้องตามหลักการแพทย์ไทย '
        "- หากไม่ใช่เหตุฉุกเฉิน: เริ่มต้นด้วย 'สถานการณ์นี้ไม่ใช่เหตุฉุกเฉิน' และให้คำแนะนำเบื้องต้นที่เหมาะสม "
        '- ข้อกำหนด: ห้ามใช้เครื่องหมายดอกจัน (*) หรือสัญลักษณ์พิเศษ ให้ใช้เพียงลำดับตัวเลข 1, 2, 3 เท่านั้น '
        'severity: วิเคราะห์ระดับความรุนแรง เลือกเพียงหนึ่งค่า: [\"critical\", \"mild\", \"none\"]. '
        'facility_type: วิเคราะห์ระดับความรุนแรง เลือกเพียงหนึ่งค่า: [\"hospital\", \"clinic\", \"none\"]. '
        'ห้ามใส่เครื่องหมายดอกจัน (*) ในคำตอบ. ห้ามใส่คำอธิบายอื่นๆ นอกเหนือจาก JSON.'
    )

def make_target_json(row: Dict[str, Any]) -> str:
    guidance = str(row.get('guidance', '') or '').strip()
    severity = normalize_severity(row.get('severity', 'none'))
    facility = normalize_facility(row.get('facility_type', 'none'))

    if not guidance:
        output_text = str(row.get('output', '') or '').strip()
        if output_text:
            if '|' in output_text:
                guidance = output_text.split('|', 1)[0].strip()
            else:
                try:
                    parsed = json.loads(output_text)
                    if isinstance(parsed, dict):
                        guidance = str(parsed.get('guidance', '') or '').strip()
                        severity = normalize_severity(parsed.get('severity', severity))
                        facility = normalize_facility(parsed.get('facility_type', facility))
                except Exception:
                    guidance = output_text

    if not guidance:
        guidance = 'สถานการณ์นี้ไม่ใช่เหตุฉุกเฉิน 1. ประเมินอาการ 2. ติดตามอาการ 3. หากแย่ลงให้โทร 1669'

    target = {
        'guidance': guidance,
        'severity': severity,
        'facility_type': facility,
    }
    return json.dumps(target, ensure_ascii=False)

def format_chat_text(input_text: str, target_json: str) -> str:
    user_content = build_user_prompt(input_text=input_text)
    return (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
        f'{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'
        f'{user_content}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
        f'{target_json}<|eot_id|>'
    )

def format_dataset_batch(examples: Dict[str, Any]) -> Dict[str, Any]:
    inputs = examples.get('input', [])
    texts = []
    for i in range(len(inputs)):
        row = {k: examples[k][i] for k in examples.keys()}
        input_text = str(row.get('input', '') or '').strip()
        if not input_text:
            texts.append('')
            continue
        target_json = make_target_json(row)
        texts.append(format_chat_text(input_text=input_text, target_json=target_json))
    return {'text': texts}

def load_and_prepare_dataset(path: str, eval_ratio: float, seed: int) -> Tuple[Dataset, Optional[Dataset]]:
    ds = load_dataset('json', data_files={'train': path}, split='train')
    if len(ds) == 0:
        raise RuntimeError(f'Dataset is empty: {path}')

    ds = ds.filter(lambda x: str(x.get('input', '') or '').strip() != '')
    ds = ds.map(format_dataset_batch, batched=True, desc='Formatting chat dataset')
    ds = ds.filter(lambda x: str(x.get('text', '') or '').strip() != '')

    if len(ds) < 20 or eval_ratio <= 0.0:
        return ds, None

    split = ds.train_test_split(test_size=eval_ratio, seed=seed, shuffle=True)
    return split['train'], split['test']


In [ ]:
# 6) Load dataset + quick preview
train_ds, eval_ds = load_and_prepare_dataset(DATASET_PATH, EVAL_RATIO, SEED)
print('Train samples:', len(train_ds))
print('Eval samples:', 0 if eval_ds is None else len(eval_ds))
print('\nSample formatted text:\n')
print(train_ds[0]['text'][:1500])


In [ ]:
# 7) Load base model + apply LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
print('Model ready')


In [ ]:
# 8) Build trainer (compatible + single-process to avoid pickle errors)
import os
import inspect

# Guard: if this cell is run before config cell
if 'WARMUP_STEPS' not in globals():
    WARMUP_STEPS = 50

# Avoid multiprocessing/pickling issues on some Colab + TRL builds
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

eval_strategy = 'steps' if eval_ds is not None else 'no'

base_training_kwargs = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=max(1, BATCH_SIZE),
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=(eval_ds is not None),
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim='adamw_8bit',
    lr_scheduler_type='cosine',
    seed=SEED,
    report_to='none',
    dataloader_num_workers=0,
)

# Prefer SFTConfig when available (newer TRL)
try:
    from trl import SFTConfig
    sft_cfg_params = inspect.signature(SFTConfig.__init__).parameters
    sft_cfg_kwargs = dict(base_training_kwargs)
    if 'evaluation_strategy' in sft_cfg_params:
        sft_cfg_kwargs['evaluation_strategy'] = eval_strategy
    elif 'eval_strategy' in sft_cfg_params:
        sft_cfg_kwargs['eval_strategy'] = eval_strategy
    if 'dataset_num_proc' in sft_cfg_params:
        sft_cfg_kwargs['dataset_num_proc'] = 1
    if 'packing' in sft_cfg_params:
        sft_cfg_kwargs['packing'] = False

    training_args = SFTConfig(**sft_cfg_kwargs)
except Exception:
    # Fallback for older TRL using plain TrainingArguments
    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    ta_kwargs = dict(base_training_kwargs)
    if 'evaluation_strategy' in ta_params:
        ta_kwargs['evaluation_strategy'] = eval_strategy
    else:
        ta_kwargs['eval_strategy'] = eval_strategy
    training_args = TrainingArguments(**ta_kwargs)

sft_params = inspect.signature(SFTTrainer.__init__).parameters
sft_kwargs = dict(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=training_args,
)
if 'tokenizer' in sft_params:
    sft_kwargs['tokenizer'] = tokenizer
elif 'processing_class' in sft_params:
    sft_kwargs['processing_class'] = tokenizer
if 'dataset_text_field' in sft_params:
    sft_kwargs['dataset_text_field'] = 'text'
if 'max_seq_length' in sft_params:
    sft_kwargs['max_seq_length'] = MAX_SEQ_LENGTH
if 'dataset_num_proc' in sft_params:
    sft_kwargs['dataset_num_proc'] = 1
if 'packing' in sft_params:
    sft_kwargs['packing'] = False

trainer = SFTTrainer(**sft_kwargs)
print('Trainer ready')


In [ ]:
# 9) Train
trainer.train()

In [ ]:
# 10) Save LoRA adapter + tokenizer
adapter_dir = os.path.join(OUTPUT_DIR, 'adapter')
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('Saved adapter to:', adapter_dir)

In [ ]:
# 11) Optional GGUF export
if SAVE_GGUF:
    gguf_dir = os.path.join(OUTPUT_DIR, 'gguf_q4_k_m')
    os.makedirs(gguf_dir, exist_ok=True)
    model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method='q4_k_m')
    print('Saved GGUF to:', gguf_dir)
else:
    print('Skipped GGUF export (SAVE_GGUF=False)')

## Notes
- ถ้าเจอ OOM ให้ลด `BATCH_SIZE` หรือ `MAX_SEQ_LENGTH`
- ถ้าอยากเทรนไวขึ้นชั่วคราว ให้ตั้ง `MAX_STEPS` เป็นค่าต่ำ เช่น 100 เพื่อ smoke test
- สำหรับงานจริงให้ใช้ epoch-based (`MAX_STEPS = -1`)